# RNA-seq Pipeline practice

We are going to use real RNA-seq data from the following publication: https://www.cell.com/heliyon/fulltext/S2405-8440(24)00413-4. GEO accession number: GSE250471

This data comes from a human macrophages cell line (U937). The data is available on GEO (count matrix) and also SRA (FASTQ files).
However, To reduce the computing time, we will be working only with reads that align to the chromosome 22.



1. First we are going to download and install all the necessary softwares and data to perform the analysis:

        a) FASTQC (quality control)

        b) Trimmomatic (trimming)

        c) STAR (alignment)

        d) samtools (BAM filtering)

        e) FeatureCounts (generating count matrix)

        f) GTF file with genome annotations and chr22 Hg38 reference fasta file

        g) Generating the STAR index for chr22

        h) Cloning the repository that contains the fastq file

2. We will perform quality control of raw FASTQ files using FASTQC

3. Trimming using Trimmomatic and post-trimming QC

4. Alignment using STAR

5. Indexing and filtering the BAM file

6. Generating count matrix with FeatureCounts

7. Loading the count matrix in python and checking the top hits


# a) downloading FASTQC

In [1]:
# Download FastQC
%%capture
!wget https://www.bioinformatics.babraham.ac.uk/projects/fastqc/fastqc_v0.12.1.zip
!unzip fastqc_v0.12.1.zip
!chmod +x FastQC/fastqc

# b) Downloading Trimmomatic

In [2]:
# Download Trimmomatic
%%capture
!wget http://www.usadellab.org/cms/uploads/supplementary/Trimmomatic/Trimmomatic-0.39.zip
!unzip Trimmomatic-0.39.zip
!java -jar Trimmomatic-0.39/trimmomatic-0.39.jar

# c) Downloading STAR

In [3]:
%%capture
!git clone https://github.com/alexdobin/STAR.git
!./STAR/bin/Linux_x86_64_static/STAR --version

#d) Installing samtools

In [4]:
# Update and install samtools
%%capture
!apt-get update
!apt-get install -y samtools
!samtools --version

# e) Installing FeatureCounts (subread)

In [5]:
%%capture
!apt-get update
!apt-get install -y subread
!featureCounts -v

# f) Downloading the chr22 of the human reference genome hg38 (USCSC) and also the GTF file that contains the genes and their location (annotation file).


In [6]:
%%capture
!wget http://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz
!gunzip chr22.fa.gz
!wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/genes/hg38.ensGene.gtf.gz
!gunzip hg38.ensGene.gtf.gz

# g) Generating the STAR index for chr22

We need to generate an index of the reference fasta for alignment

In this step we need to include a parameter: --sjdbOverhang $read_length -1, which is related to the kmers STAR is going to use. for this number, we used read length -1

In [ ]:
!./STAR/bin/Linux_x86_64_static/STAR --runThreadN 6 --runMode genomeGenerate --genomeDir Gencode_STAR_Index --genomeFastaFiles chr22.fa --sjdbGTFfile hg38.ensGene.gtf --sjdbOverhang 49 --genomeSAindexNbases 11

	./STAR/bin/Linux_x86_64_static/STAR --runThreadN 6 --runMode genomeGenerate --genomeDir Gencode_STAR_Index --genomeFastaFiles chr22.fa --sjdbGTFfile hg38.ensGene.gtf --sjdbOverhang 49 --genomeSAindexNbases 11
	STAR version: 2.7.11b   compiled: 2024-01-25T16:12:02-05:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Sep 30 19:31:04 ..... started STAR run
Sep 30 19:31:04 ... starting to generate Genome files
Sep 30 19:31:05 ..... processing annotations GTF
Sep 30 19:31:12 ... starting to sort Suffix Array. This may take a long time...
Sep 30 19:31:13 ... sorting Suffix Array chunks and saving them to disk...
Sep 30 19:32:01 ... loading chunks from disk, packing SA...
Sep 30 19:32:03 ... finished generating suffix array
Sep 30 19:32:03 ... generating Suffix Array index
Sep 30 19:32:07 ... completed Suffix Array index
Sep 30 19:32:07 ..... inserting junctions into the genome indices
Sep 30 19:32:12 ... writing Genome to disk ...
Sep 30 19:32:12 ... writing Suffix Array to disk ...
Sep

# h) Cloning the repository that contains the fastq file

Now we need to get the raw data for our analysis. Usually we will just download it from SRA but because we don't want to overwhelm google colab, we will use a file I filtered for chr22 reads in the github repository

In [7]:
%%capture
!git clone https://github.com/almejiaga/RNA-seq-bootcamp-day-2.git
!gunzip RNA-seq-bootcamp-day-2/Exercises/data/unique_chr22.fastq.gz

# 2. Running FASTQC to get the QC metrics we saw in the lecture

In [ ]:
!FastQC/fastqc RNA-seq-bootcamp-day-2/Exercises/data/unique_chr22.fastq

null
Started analysis of unique_chr22.fastq
Approx 5% complete for unique_chr22.fastq
Approx 10% complete for unique_chr22.fastq
Approx 15% complete for unique_chr22.fastq
Approx 20% complete for unique_chr22.fastq
Approx 25% complete for unique_chr22.fastq
Approx 30% complete for unique_chr22.fastq
Approx 35% complete for unique_chr22.fastq
Approx 40% complete for unique_chr22.fastq
Approx 45% complete for unique_chr22.fastq
Approx 50% complete for unique_chr22.fastq
Approx 55% complete for unique_chr22.fastq
Approx 60% complete for unique_chr22.fastq
Approx 65% complete for unique_chr22.fastq
Approx 70% complete for unique_chr22.fastq
Approx 75% complete for unique_chr22.fastq
Approx 80% complete for unique_chr22.fastq
Approx 85% complete for unique_chr22.fastq
Approx 90% complete for unique_chr22.fastq
Approx 95% complete for unique_chr22.fastq
Analysis complete for unique_chr22.fastq


In [ ]:
!ls RNA-seq-bootcamp-day-2/Exercises/data/

data.txt  unique_chr22.fastq  unique_chr22_fastqc.html	unique_chr22_fastqc.zip


# Let's download the file to our personal computer to visualize the html file better

In [ ]:
from google.colab import files
files.download('RNA-seq-bootcamp-day-2/Exercises/data/unique_chr22_fastqc.html')  # or .csv, .html, .pdf, etc.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls FastQC/fastqc RNA-seq-bootcamp-day-2/Exercises/data/

FastQC/fastqc

RNA-seq-bootcamp-day-2/Exercises/data/:
data.txt  unique_chr22.fastq  unique_chr22_fastqc.html	unique_chr22_fastqc.zip


# 3.1 Trimming our raw reads to ensure the best quality for aligment

In [ ]:
!java -jar Trimmomatic-0.39/trimmomatic-0.39.jar SE \
  -threads 4 \
  RNA-seq-bootcamp-day-2/Exercises/data/unique_chr22.fastq \
  R1_trimmed.fastq \
  ILLUMINACLIP:Trimmomatic-0.39/adapters/TruSeq3-SE.fa:2:30:10 \
  LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36


TrimmomaticSE: Started with arguments:
 -threads 4 RNA-seq-bootcamp-day-2/Exercises/data/unique_chr22.fastq R1_trimmed.fastq ILLUMINACLIP:Trimmomatic-0.39/adapters/TruSeq3-SE.fa:2:30:10 LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
Using Long Clipping Sequence: 'AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGTA'
Using Long Clipping Sequence: 'AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC'
ILLUMINACLIP: Using 0 prefix pairs, 2 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Quality encoding detected as phred33
Input Reads: 674468 Surviving: 672201 (99.66%) Dropped: 2267 (0.34%)
TrimmomaticSE: Completed successfully


# 3.2 Running FASTQC after trimming to see if the QC metric have improved or not

In [ ]:
!FastQC/fastqc R1_trimmed.fastq

null
Started analysis of R1_trimmed.fastq
Approx 5% complete for R1_trimmed.fastq
Approx 10% complete for R1_trimmed.fastq
Approx 15% complete for R1_trimmed.fastq
Approx 20% complete for R1_trimmed.fastq
Approx 25% complete for R1_trimmed.fastq
Approx 30% complete for R1_trimmed.fastq
Approx 35% complete for R1_trimmed.fastq
Approx 40% complete for R1_trimmed.fastq
Approx 45% complete for R1_trimmed.fastq
Approx 50% complete for R1_trimmed.fastq
Approx 55% complete for R1_trimmed.fastq
Approx 60% complete for R1_trimmed.fastq
Approx 65% complete for R1_trimmed.fastq
Approx 70% complete for R1_trimmed.fastq
Approx 75% complete for R1_trimmed.fastq
Approx 80% complete for R1_trimmed.fastq
Approx 85% complete for R1_trimmed.fastq
Approx 90% complete for R1_trimmed.fastq
Approx 95% complete for R1_trimmed.fastq
Analysis complete for R1_trimmed.fastq


# Once again, we want to download the html file to visualize it

In [ ]:
from google.colab import files
files.download('R1_trimmed_fastqc.html')  # or .csv, .html, .pdf, etc.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 4.  Alignment

After trimming the FASTQ and keeping only the high quality reads, we are ready to align to the human genome. we need to specify the folder that contains the index for the reference. There are a lot of other parameters associated with the alignment that we play with.

In [ ]:
!./STAR/bin/Linux_x86_64_static/STAR --genomeDir Gencode_STAR_Index --runThreadN 1 --readFilesIn R1_trimmed.fastq --outSAMtype BAM SortedByCoordinate --limitBAMsortRAM 17179869184 --outFilterType BySJout --outFilterMultimapNmax 20 --alignSJoverhangMin 8 --alignSJDBoverhangMin 1 --outFilterMismatchNmax 999 --outFilterMismatchNoverLmax 0.04 --alignIntronMin 20 --alignIntronMax 1000000 --seedSearchStartLmax 30 --outFileNamePrefix 1_mapped_chr22

	./STAR/bin/Linux_x86_64_static/STAR --genomeDir Gencode_STAR_Index --runThreadN 1 --readFilesIn R1_trimmed.fastq --outSAMtype BAM SortedByCoordinate --limitBAMsortRAM 17179869184 --outFilterType BySJout --outFilterMultimapNmax 20 --alignSJoverhangMin 8 --alignSJDBoverhangMin 1 --outFilterMismatchNmax 999 --outFilterMismatchNoverLmax 0.04 --alignIntronMin 20 --alignIntronMax 1000000 --seedSearchStartLmax 30 --outFileNamePrefix 1_mapped_chr22
	STAR version: 2.7.11b   compiled: 2024-01-25T16:12:02-05:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Sep 30 19:40:12 ..... started STAR run
Sep 30 19:40:12 ..... loading genome
Sep 30 19:40:13 ..... started mapping
Sep 30 19:42:37 ..... finished mapping
Sep 30 19:42:37 ..... started sorting BAM
Sep 30 19:42:40 ..... finished successfully


# 5. Indexing and filtering out reads with low mapping quality scores

In [ ]:

!samtools index 1_mapped_chr22Aligned.sortedByCoord.out.bam
!samtools view -F 2820 -q 30 -@ 4 -b 1_mapped_chr22Aligned.sortedByCoord.out.bam > 1_filtered.bam

# 6. Generaring the count matrix



In [ ]:
!featureCounts -T 4 -t exon -g gene_id -a hg38.ensGene.gtf -o one_sample.txt 1_filtered.bam


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.3

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 1 BAM file                                       ||
||                                                                            ||
||                           1_filtered.bam                                   ||
||                                                                            ||
||             Output file : one_sample.txt                      

# 7. Loading the count matrix in python and checking the top hits

If we search the top genes, they should be related to constitutive genes or macrophages-related genes

In [ ]:
import pandas as pd

# Read the featureCounts output, skip header lines
df = pd.read_csv("one_sample.txt", sep="\t", comment="#")

# Get the BAM column name (it usually matches your BAM filename)
bam_col = "1_filtered.bam"

# Sort by counts in descending order and show top 10
top10 = df.sort_values(by=bam_col, ascending=False).head(10)

top10

,Geneid,Chr,Start,End,Strand,Length,1_filtered.bam
38710,ENSG00000128340,chr22;chr22;chr22;chr22;chr22;chr22;chr22;chr2...,37225261;37225852;37225966;37226671;37226671;3...,37226039;37226039;37226039;37226803;37226803;3...,-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;...,2532,27818
38858,ENSG00000213857,chr22,41074180,41075239,-,1060,23101
37804,ENSG00000241838,chr22,15721486,15722608,+,1123,21142
38673,ENSG00000100345,chr22;chr22;chr22;chr22;chr22;chr22;chr22;chr2...,36281281;36284093;36284162;36284214;36284403;3...,36282785;36284265;36284307;36284511;36284511;3...,-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;...,8959,19757
38727,ENSG00000100097,chr22;chr22;chr22;chr22;chr22;chr22;chr22;chr2...,37675608;37675652;37675657;37675682;37675688;3...,37675711;37675711;37675711;37675711;37675711;3...,+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+,1099,13966
38809,ENSG00000100316,chr22;chr22;chr22;chr22;chr22;chr22;chr22;chr2...,39312882;39312884;39312884;39312884;39312884;3...,39312984;39313310;39312984;39312984;39312984;3...,-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;...,4787,9528
38575,ENSG00000232346,chr22,32039490,32039896,+,407,9274
38511,ENSG00000225774,chr22,30542536,30544305,+,1770,6603
38995,ENSG00000100364,chr22;chr22;chr22;chr22;chr22;chr22;chr22;chr2...,45190338;45195828;45195828;45195828;45196038;4...,45197216;45197216;45197216;45197216;45197216;4...,-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;...,10019,6178
37860,ENSG00000177663,chr22;chr22;chr22;chr22;chr22;chr22;chr22;chr2...,17084954;17084959;17084959;17085057;17097062;1...,17085229;17085229;17085229;17085229;17097509;1...,+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;...,9036,5974
